# Causal direction: Mendelian randomisation of the GrimAge proteins

Displays the MR results written by the R scripts in `mr/` (`results/mr/`) and what they changed in the agent's assessments. The write-up is `docs/3-causal-direction.md`.

In [1]:
import json

import pandas as pd
from IPython.display import Image, Markdown, display

from agetarget import config as cfg

pd.set_option("display.precision", 3)
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 90)

from agetarget.targets import agent

M = cfg.MR_RESULTS
T = cfg.TARGET_RESULTS

## 1. Instruments

One pQTL study per protein; cis instruments are variants at p < 5e-8 within 1 Mb of the gene.

In [2]:
exposures = pd.read_csv(M / "mr_exposures.csv")
display(exposures)
pd.read_csv(M / "mr_instruments.csv")[["exposure", "SNP", "effect_allele.exposure", "beta.exposure", "pval.exposure",
                                        "samplesize.exposure", "F_stat"]].round(3)

,protein,exposure_id,study,n_genome_wide,n_cis
0,ADM,ebi-a-GCST90011996,"Folkersen L 2020 (ebi-a-GCST90011996, n = 21,758)",5,2
1,B2M,prot-c-3485_28_2,Suhre K 2019 (prot-c-3485_28_2),1,0
2,CST3,ebi-a-GCST90025945,"Barton AR 2021 (ebi-a-GCST90025945, n = 437,846)",372,6
3,SERPINE1,ebi-a-GCST90019422,"Pietzner M 2020 (ebi-a-GCST90019422, n = 10,708)",3,1
4,GDF15,ebi-a-GCST90011998,"Folkersen L 2020 (ebi-a-GCST90011998, n = 21,758)",3,2
5,LEP,ebi-a-GCST90012076,"Folkersen L 2020 (ebi-a-GCST90012076, n = 21,758)",2,1
6,TIMP1,prot-c-2211_9_6,Suhre K 2019 (prot-c-2211_9_6),0,0


,exposure,SNP,effect_allele.exposure,beta.exposure,pval.exposure,samplesize.exposure,F_stat
0,ADM,rs11042731,T,0.154,0.0,21758,185.731
1,ADM,rs2923084,G,-0.080,0.0,21758,37.668
2,CST3,rs1064039,T,-0.408,0.0,437846,32239.938
3,CST3,rs73102389,A,0.138,0.0,437846,3162.068
4,CST3,rs118005442,T,0.069,0.0,437846,307.169
5,CST3,rs74335775,T,0.081,0.0,437846,200.589
6,CST3,rs6113974,A,0.057,0.0,437846,152.028
7,CST3,rs41282288,T,0.035,0.0,437846,30.473
8,SERPINE1,rs2227631,G,-0.103,0.0,10708,61.124
9,GDF15,rs1227734,T,0.369,0.0,21758,805.250


## 2. MR estimates

Primary estimate per protein and outcome (Wald ratio for one variant, inverse-variance weighted for several), Bonferroni-corrected over all primary tests. Every outcome is coded so that a positive b means worse ageing.

In [3]:
res = pd.read_csv(M / "mr_results.csv")
primary = res[res.primary].sort_values("pval")
table = primary[["protein", "outcome", "method", "nsnp", "b", "se", "pval", "p_bonferroni", "direction"]].round(3)
table[["pval", "p_bonferroni"]] = primary[["pval", "p_bonferroni"]].map(lambda v: f"{v:.2g}")
display(table.set_index("protein"))
print("Heterogeneity where a protein has several instruments:")
pd.read_csv(M / "mr_checks.csv")[["protein", "outcome", "method", "Q", "Q_pval"]].dropna(subset=["Q"]).round(3)

,outcome,method,nsnp,b,se,pval,p_bonferroni,direction
protein,,,,,,,,
SERPINE1,parental_longevity,Wald ratio,1,0.070,0.022,0.0015,0.03,higher protein -> worse ageing (hypothesis: inhibit)
LEP,parental_longevity,Wald ratio,1,0.045,0.030,0.14,1,higher protein -> worse ageing (hypothesis: inhibit)
ADM,frailty,Inverse variance weighted,2,0.026,0.022,0.24,1,higher protein -> worse ageing (hypothesis: inhibit)
LEP,frailty,Wald ratio,1,0.050,0.045,0.26,1,higher protein -> worse ageing (hypothesis: inhibit)
GDF15,frailty,Inverse variance weighted,2,0.023,0.022,0.28,1,higher protein -> worse ageing (hypothesis: inhibit)
ADM,parental_longevity,Inverse variance weighted,2,0.015,0.015,0.3,1,higher protein -> worse ageing (hypothesis: inhibit)
LEP,grimage_accel_a,Wald ratio,1,-0.401,0.420,0.34,1,higher protein -> better ageing (hypothesis: activate)
SERPINE1,grimage_accel_b,Wald ratio,1,0.261,0.325,0.42,1,higher protein -> worse ageing (hypothesis: inhibit)
SERPINE1,frailty,Wald ratio,1,0.025,0.033,0.44,1,higher protein -> worse ageing (hypothesis: inhibit)


Heterogeneity where a protein has several instruments:


,protein,outcome,method,Q,Q_pval
0,ADM,parental_longevity,Inverse variance weighted,0.418,0.518
1,ADM,frailty,Inverse variance weighted,0.176,0.675
2,ADM,grimage_accel_a,Inverse variance weighted,0.921,0.337
3,ADM,grimage_accel_b,Inverse variance weighted,0.899,0.343
4,CST3,parental_longevity,MR Egger,3.497,0.478
5,CST3,parental_longevity,Inverse variance weighted,3.504,0.623
7,CST3,frailty,MR Egger,4.719,0.317
8,CST3,frailty,Inverse variance weighted,4.754,0.447
10,CST3,grimage_accel_a,MR Egger,0.549,0.969
11,CST3,grimage_accel_a,Inverse variance weighted,0.872,0.972


## 3. Follow-up of PAI-1 (SERPINE1)

The one Bonferroni-significant result: genetically higher PAI-1, shorter parental lifespan. Its instrument (rs2227631) in other PAI-1 studies, colocalisation at the locus, and other traits linked to the variant.

In [4]:
display(pd.read_csv(M / "replication_rs2227631.csv")[["study", "measure", "ea", "beta", "se", "p", "n"]])
coloc = pd.read_csv(next(M.glob("coloc_*.csv")))
display(coloc[["protein", "outcome", "flank_kb", "nsnps", "PP.H1.abf", "PP.H3.abf", "PP.H4.abf", "top_snp",
               "top_snp_pp_h4", "outcome_top_snp", "outcome_min_p"]].round(3))
pd.read_csv(M / "phewas_rs2227631.csv").sort_values("p")[["trait", "id", "ea", "beta", "p"]].head(10)

,study,measure,ea,beta,se,p,n
0,gilly2020,protein,G,-0.102,0.041,1.226e-02,1323
1,dnam_pai1_mccartney,DNAm surrogate,G,0.056,0.074,4.533e-01,34448
2,pietzner2020_somascan,protein,G,-0.064,0.013,1.293e-06,10708
3,sun2018_somascan,protein,G,-0.036,0.025,1.479e-01,3301


,protein,outcome,flank_kb,nsnps,PP.H1.abf,PP.H3.abf,PP.H4.abf,top_snp,top_snp_pp_h4,outcome_top_snp,outcome_min_p
0,SERPINE1,parental_longevity,250,2290,0.832,0.036,0.132,rs2227631,0.939,rs4268055,0.001


,trait,id,ea,beta,p
0,Plasminogen activator inhibitor 1 measurement,ebi-a-GCST90019422,G,-0.103,6.439e-15
1,ENSG00000087077,eqtl-a-ENSG00000087077,G,-0.078,8.734e-11
2,Tissue-type plasminogen activator measurement,ebi-a-GCST90019414,G,-0.077,1.069e-09
3,ENSG00000196411,eqtl-a-ENSG00000196411,G,-0.071,3.742e-09
4,ENSG00000106366,eqtl-a-ENSG00000106366,G,-0.069,1.066e-08
5,"Pulse rate, automated reading",ukb-b-18103,G,0.012,3.600e-08
6,ENSG00000227053,eqtl-a-ENSG00000227053,G,-0.057,2.075e-06
7,Red blood cell count,ebi-a-GCST90002363,G,0.009,6.290e-06
8,ENSG00000239521,eqtl-a-ENSG00000239521,G,-0.054,6.744e-06


## 4. What the MR evidence changed in the agent

The same model and evidence, with (version 3) and without (version 2) the `get_causal_evidence` tool.

In [5]:
def modulation(batch):
    calls = {}
    for r in agent.load_results(batch):
        if r.assessment:
            calls.setdefault(r.target, []).append(r.assessment["proposed_modulation"])
    return pd.Series({t: max(set(v), key=v.count) for t, v in calls.items()})

v2 = pd.read_csv(T / "rebuilt" / "20261116-230637" / "ranking.csv", index_col=0)
v3 = pd.read_csv(T / "ranking.csv", index_col=0)
latest = sorted(p for p in cfg.AGENT_RUNS.iterdir() if p.is_dir())[-1]
pd.DataFrame({
    "v2 score": v2.agent_overall, "v3 score": v3.agent_overall,
    "v2 recommendations": v2.recommendation_counts, "v3 recommendations": v3.recommendation_counts,
    "v2 modulation": modulation(cfg.AGENT_RUNS / "20261116-230637"), "v3 modulation": modulation(latest),
}).loc[v3.index].round(2)

,v2 score,v3 score,v2 recommendations,v3 recommendations,v2 modulation,v3 modulation
SERPINE1,3.37,3.37,consider 3,"prioritise 2, consider 1",inhibit,inhibit
ADM,3.10,3.20,consider 3,consider 3,unclear,unclear
GDF15,2.70,2.70,consider 3,consider 3,unclear,unclear
B2M,2.70,2.58,consider 3,consider 3,unclear,unclear
TIMP1,2.50,2.30,"deprioritise 2, consider 1","deprioritise 2, consider 1",unclear,unclear
CST3,2.85,2.25,"deprioritise 2, consider 1",deprioritise 3,unclear,unclear
LEP,2.30,1.95,deprioritise 3,deprioritise 3,unclear,unclear
